## NB_patient_360
Builds `<catalog>.gold.patient_360` from the silver layer, following STTM `sttm_silver_to_gold.xlsx` (sheet: patient_360).

* **Grain:** one row per patient
* **Driving table:** `silver.patients`
* **Joins:** LEFT JOIN `silver.hospitals` (registered_hospital_id = hospital_id), LEFT JOIN `silver.insurance_providers` (insurance_provider_id), LEFT JOIN `silver.lab_results` aggregated by patient_id
* **Filters:** none (all patients included)
* **Load type:** FULL - `DataFrame.writeTo(...).createOrReplace()` (PySpark equivalent of CREATE OR REPLACE TABLE ... AS SELECT)

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "dbw_caresync")
catalog = dbutils.widgets.get("catalog").strip() or "dbw_caresync"

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"
print(f"[SOURCE SCHEMA] {source_schema}")
print(f"[TARGET TABLE] {target_table}")

In [0]:
print("[STEP 1] Reading silver sources")
patients_df = spark.table(f"{source_schema}.patients").alias("p")
hospitals_df = spark.table(f"{source_schema}.hospitals").alias("h")
insurance_df = spark.table(f"{source_schema}.insurance_providers").alias("ip")
lab_results_df = spark.table(f"{source_schema}.lab_results")


def status_flag(status):
    """1 when result_status equals the given value, else 0 (used for conditional counts)."""
    return F.when(F.col("result_status") == status, 1).otherwise(0)


print("[STEP 2] Aggregating lab_results per patient")
lab_agg_df = (
    lab_results_df.groupBy("patient_id")
    .agg(
        F.count("*").alias("total_tests"),
        F.sum(status_flag("Abnormal")).alias("abnormal_test_count"),
        F.sum(status_flag("Critical")).alias("critical_test_count"),
        F.max("test_date").alias("last_test_date"),
    )
    .alias("la")
)

print("[STEP 3] Joining (LEFT from patients) and applying column mapping")
dob = F.col("p.date_of_birth").cast("date")

patient_360_df = (
    patients_df
    .join(hospitals_df, F.col("p.registered_hospital_id") == F.col("h.hospital_id"), "left")
    .join(insurance_df, F.col("p.insurance_provider_id") == F.col("ip.insurance_provider_id"), "left")
    .join(lab_agg_df, F.col("p.patient_id") == F.col("la.patient_id"), "left")
    .select(
        F.col("p.patient_id").cast("int").alias("patient_id"),
        F.col("p.first_name").cast("string").alias("first_name"),
        F.col("p.last_name").cast("string").alias("last_name"),
        dob.alias("date_of_birth"),
        F.floor(F.datediff(F.current_date(), dob) / 365.25).cast("int").alias("age"),
        F.col("p.gender").cast("string").alias("gender"),
        F.col("p.city").cast("string").alias("city"),
        F.col("p.state").cast("string").alias("state"),
        F.col("p.registered_hospital_id").cast("int").alias("hospital_id"),
        F.col("h.hospital_name").cast("string").alias("hospital_name"),
        F.col("p.insurance_provider_id").cast("int").alias("insurance_provider_id"),
        F.col("ip.provider_name").cast("string").alias("insurance_provider_name"),
        F.coalesce(F.col("la.total_tests"), F.lit(0)).cast("int").alias("total_tests"),
        F.coalesce(F.col("la.abnormal_test_count"), F.lit(0)).cast("int").alias("abnormal_test_count"),
        F.coalesce(F.col("la.critical_test_count"), F.lit(0)).cast("int").alias("critical_test_count"),
        F.col("la.last_test_date").cast("date").alias("last_test_date"),
    )
)

print(f"[STEP 4] Writing {target_table} (full rebuild)")
patient_360_df.writeTo(target_table).using("delta").createOrReplace()
spark.sql(
    f"COMMENT ON TABLE {target_table} IS "
    "'Gold: one row per patient with hospital, insurance and lab-result summary. Built per STTM silver-to-gold v1.0.'"
)
print(f"[STEP 4 COMPLETE] {target_table} rebuilt")

In [0]:
print("[STEP 5] Validating grain")
src_count = spark.table(f"{source_schema}.patients").count()
stats = (
    spark.table(target_table)
    .agg(F.count("*").alias("row_count"), F.countDistinct("patient_id").alias("key_count"))
    .first()
)
print(f"[SOURCE PATIENTS] {src_count} | [TARGET ROWS] {stats.row_count} | [DISTINCT patient_id] {stats.key_count}")

if stats.row_count != stats.key_count:
    raise ValueError("Grain violation: duplicate patient_id in patient_360")
if stats.row_count != src_count:
    raise ValueError("Row count mismatch: LEFT JOIN policy requires one row per source patient")

print("[STEP 5 COMPLETE] Validation passed")

print("[STEP 6] Computing processed record count")
record_count = patient_360_df.count()
print(f"[STEP 6 COMPLETE] Processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))